In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_excel("../data/raw/online_retail.xlsx")

In [3]:
print(f"Rows       : {df.shape[0]:,}")
print(f"Columns    : {df.shape[1]}")
print(f"Memory     : {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

Rows       : 541,909
Columns    : 8
Memory     : 126.18 MB


In [4]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [5]:
df_clean = df.copy()

## Remove Exact Duplicate Rows

During data profiling, 5,268 exact duplicate rows were identified, representing approximately 0.97% of the dataset.

Because these rows contain identical values across all columns, they are treated as duplicate records rather than separate business transactions.

Therefore, exact duplicate rows will be removed before further analysis.

In [6]:
duplicate_count = df_clean.duplicated().sum()

print(f"Duplicate rows found: {duplicate_count:,}")

Duplicate rows found: 5,268


In [7]:
df_clean = df_clean.drop_duplicates().copy()

In [8]:
print(f"Rows after removing duplicates: {len(df_clean):,}")
print(f"Remaining duplicates: {df_clean.duplicated().sum():,}")

Rows after removing duplicates: 536,641
Remaining duplicates: 0


In [9]:
rows_before = len(df)

print(f"Rows before cleaning : {rows_before:,}")
print(f"Rows after cleaning  : {len(df_clean):,}")
print(f"Rows removed         : {rows_before - len(df_clean):,}")

Rows before cleaning : 541,909
Rows after cleaning  : 536,641
Rows removed         : 5,268


## Handle Cancellation Transactions

In the profiling stage, transactions with an `InvoiceNo` starting with `C` were identified as cancellation transactions.

Cancellation transactions are not treated as normal sales because their negative quantities can distort sales metrics such as revenue, units sold, and average order value.

Therefore, cancellation status will be identified using the `InvoiceNo` prefix rather than relying solely on negative quantity.

In [10]:
df_clean["is_cancelled"] = df_clean["InvoiceNo"].astype(str).str.startswith("C")

In [12]:
df_clean["is_cancelled"].value_counts()

is_cancelled
False    527390
True       9251
Name: count, dtype: int64

In [13]:
cancellation_summary = (
    df_clean["is_cancelled"]
    .value_counts()
    .rename(index={False: "Valid", True: "Cancelled"})
)

cancellation_summary

is_cancelled
Valid        527390
Cancelled      9251
Name: count, dtype: int64

In [14]:
cancelled_count = df_clean["is_cancelled"].sum()
cancelled_percentage = cancelled_count / len(df_clean) * 100

print(f"Cancelled rows     : {cancelled_count:,}")
print(f"Cancellation rate  : {cancelled_percentage:.2f}%")

Cancelled rows     : 9,251
Cancellation rate  : 1.72%


In [15]:
df_clean.groupby("is_cancelled")["Quantity"].agg(
    ["count", "sum", "mean", "median", "min", "max"]
)

,count,sum,mean,median,min,max
is_cancelled,,,,,,
False,527390,5438062,10.311272,3.0,-9600,80995
True,9251,-275560,-29.787050,-2.0,-80995,-1


## Remove Cancellation Transactions

After validation, 9,251 transactions were identified as cancellations based on the `InvoiceNo` prefix `C`.

These transactions will be excluded from the valid sales dataset because they represent cancelled transactions rather than completed sales.

The cancellation records are retained in the raw dataset for traceability, while `df_clean` will contain only valid sales transactions for subsequent analysis.

In [16]:
rows_before_cancellation = len(df_clean)

df_clean = df_clean[~df_clean["is_cancelled"]].copy()

rows_after_cancellation = len(df_clean)

print(f"Rows before removing cancellations : {rows_before_cancellation:,}")
print(f"Cancelled rows removed              : {rows_before_cancellation - rows_after_cancellation:,}")
print(f"Rows after removing cancellations   : {rows_after_cancellation:,}")

Rows before removing cancellations : 536,641
Cancelled rows removed              : 9,251
Rows after removing cancellations   : 527,390


In [17]:
print(f"Remaining cancelled rows: {df_clean['is_cancelled'].sum():,}")

Remaining cancelled rows: 0


## Handle Missing CustomerID

CustomerID is essential for customer-level analysis, particularly RFM analysis and customer segmentation.

However, missing CustomerID does not necessarily mean that the transaction itself is invalid. The transaction may still contain valid product, quantity, price, date, and country information.

Therefore, transactions with missing CustomerID will not be immediately removed from the main cleaned dataset.

Instead, they will be excluded only from customer-level analysis such as RFM and customer segmentation.

In [18]:
missing_customer = df_clean["CustomerID"].isna().sum()
missing_customer_pct = missing_customer / len(df_clean) * 100

print(f"Missing CustomerID : {missing_customer:,}")
print(f"Percentage         : {missing_customer_pct:.2f}%")

Missing CustomerID : 134,658
Percentage         : 25.53%


In [19]:
df_clean["CustomerID"].isna().value_counts()

CustomerID
False    392732
True     134658
Name: count, dtype: int64

In [20]:
customer_status = (
    df_clean["CustomerID"]
    .isna()
    .map({True: "Missing", False: "Available"})
)

customer_status.value_counts()

CustomerID
Available    392732
Missing      134658
Name: count, dtype: int64

In [21]:
df_clean.groupby(
    df_clean["CustomerID"].isna()
)["Quantity"].agg(
    ["count", "sum", "mean", "median"]
)

,count,sum,mean,median
CustomerID,,,,
False,392732,5165886,13.153718,6.0
True,134658,272176,2.021239,1.0


In [22]:
df_clean["Revenue"] = df_clean["Quantity"] * df_clean["UnitPrice"]

df_clean.groupby(
    df_clean["CustomerID"].isna()
)["Revenue"].agg(
    ["count", "sum", "mean", "median"]
)

,count,sum,mean,median
CustomerID,,,,
False,392732,8887208.894,22.629195,12.39
True,134658,1732777.790,12.867990,4.96


## Create Customer-Level Analysis Dataset

Transactions with missing CustomerID will be retained in the main cleaned dataset because they still contain valid sales information.

However, CustomerID is required for customer-level analysis.

Therefore, a separate customer-level dataset will be created by filtering transactions with available CustomerID.

This dataset will be used for:

- RFM analysis
- Customer segmentation
- Customer behavior analysis
- Customer-level revenue analysis

In [23]:
df_customer = df_clean[df_clean["CustomerID"].notna()].copy()

In [24]:
print(f"df_clean rows    : {len(df_clean):,}")
print(f"df_customer rows : {len(df_customer):,}")

df_clean rows    : 527,390
df_customer rows : 392,732


In [25]:
print(f"Missing CustomerID in df_customer: {df_customer['CustomerID'].isna().sum():,}")

Missing CustomerID in df_customer: 0


## Handle UnitPrice Anomalies

UnitPrice contains several non-standard values, including zero, negative, and extremely high prices.

These values were investigated during the profiling stage and were found to include operational transactions such as fees, manual adjustments, bad debt adjustments, and records without normal product information.

For the sales analysis dataset, transactions with `UnitPrice <= 0` will be excluded because they do not represent normal positive-priced product sales.

Extreme positive prices will not be removed solely based on statistical outlier thresholds and will be investigated separately.

In [26]:
unitprice_summary = pd.Series({
    "UnitPrice <= 0": (df_clean["UnitPrice"] <= 0).sum(),
    "UnitPrice < 0": (df_clean["UnitPrice"] < 0).sum(),
    "UnitPrice == 0": (df_clean["UnitPrice"] == 0).sum(),
    "UnitPrice > 100": (df_clean["UnitPrice"] > 100).sum()
})

unitprice_summary

UnitPrice <= 0     2512
UnitPrice < 0         2
UnitPrice == 0     2510
UnitPrice > 100     811
dtype: int64

In [27]:
zero_or_negative_price = df_clean[df_clean["UnitPrice"] <= 0]

print(f"Rows with UnitPrice <= 0: {len(zero_or_negative_price):,}")

Rows with UnitPrice <= 0: 2,512


In [28]:
zero_or_negative_price[
    [
        "InvoiceNo",
        "StockCode",
        "Description",
        "Quantity",
        "UnitPrice",
        "CustomerID",
        "Country"
    ]
].head(20)

,InvoiceNo,StockCode,Description,Quantity,UnitPrice,CustomerID,Country
622,536414,22139,NaN,56,0.0,NaN,United Kingdom
1970,536545,21134,NaN,1,0.0,NaN,United Kingdom
1971,536546,22145,NaN,1,0.0,NaN,United Kingdom
1972,536547,37509,NaN,1,0.0,NaN,United Kingdom
1987,536549,85226A,NaN,1,0.0,NaN,United Kingdom
1988,536550,85044,NaN,1,0.0,NaN,United Kingdom
2024,536552,20950,NaN,1,0.0,NaN,United Kingdom
2025,536553,37461,NaN,3,0.0,NaN,United Kingdom
2026,536554,84670,NaN,23,0.0,NaN,United Kingdom
2406,536589,21777,NaN,-10,0.0,NaN,United Kingdom


In [29]:
zero_or_negative_price["CustomerID"].notna().value_counts()

CustomerID
False    2472
True       40
Name: count, dtype: int64

In [30]:
zero_or_negative_price["Revenue"].describe()

count     2512.000000
mean        -8.807373
std        312.071921
min     -11062.060000
25%         -0.000000
50%          0.000000
75%          0.000000
max         -0.000000
Name: Revenue, dtype: float64

In [31]:
df_clean = df_clean[df_clean["UnitPrice"] > 0].copy()

print(f"Rows after removing invalid UnitPrice: {len(df_clean):,}")
print(f"Remaining UnitPrice <= 0: {(df_clean['UnitPrice'] <= 0).sum():,}")

Rows after removing invalid UnitPrice: 524,878
Remaining UnitPrice <= 0: 0


In [32]:
df_clean["Revenue"] = df_clean["Quantity"] * df_clean["UnitPrice"]

print(df_clean["Revenue"].describe())

count    524878.000000
mean         20.275399
std         271.693566
min           0.001000
25%           3.900000
50%           9.920000
75%          17.700000
max      168469.600000
Name: Revenue, dtype: float64


In [33]:
print("Negative revenue:", (df_clean["Revenue"] < 0).sum())
print("Zero revenue:", (df_clean["Revenue"] == 0).sum())

Negative revenue: 0
Zero revenue: 0


In [34]:
missing_description = df_clean["Description"].isna().sum()
missing_description_pct = (
    missing_description / len(df_clean) * 100
)

print(f"Missing Description : {missing_description:,}")
print(f"Percentage          : {missing_description_pct:.2f}%")

Missing Description : 0
Percentage          : 0.00%


In [35]:
df_clean[df_clean["Description"].isna()]["CustomerID"].notna().value_counts()

Series([], Name: count, dtype: int64)

In [36]:
df_clean[df_clean["Description"].isna()]["Revenue"].describe()

count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: Revenue, dtype: float64

In [37]:
df_clean[df_clean["Description"].isna()].head(20)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,is_cancelled,Revenue


In [38]:
print("Rows       :", f"{len(df_clean):,}")
print("Columns    :", len(df_clean.columns))
print("Missing values:")
print(df_clean.isna().sum())

Rows       : 524,878
Columns    : 10
Missing values:
InvoiceNo            0
StockCode            0
Description          0
Quantity             0
InvoiceDate          0
UnitPrice            0
CustomerID      132186
Country              0
is_cancelled         0
Revenue              0
dtype: int64


In [39]:
print("Quantity <= 0 :", (df_clean["Quantity"] <= 0).sum())
print("UnitPrice <= 0:", (df_clean["UnitPrice"] <= 0).sum())
print("Revenue <= 0  :", (df_clean["Revenue"] <= 0).sum())

Quantity <= 0 : 0
UnitPrice <= 0: 0
Revenue <= 0  : 0


In [40]:
print("Min InvoiceDate:", df_clean["InvoiceDate"].min())
print("Max InvoiceDate:", df_clean["InvoiceDate"].max())

Min InvoiceDate: 2010-12-01 08:26:00
Max InvoiceDate: 2011-12-09 12:50:00


In [41]:
df_customer = df_clean[df_clean["CustomerID"].notna()].copy()

print("Customer transaction rows :", f"{len(df_customer):,}")
print("Missing CustomerID        :", df_customer["CustomerID"].isna().sum())
print("Unique customers          :", df_customer["CustomerID"].nunique())

Customer transaction rows : 392,692
Missing CustomerID        : 0
Unique customers          : 4338


In [42]:
print(df_customer[[
    "InvoiceNo",
    "StockCode",
    "Quantity",
    "InvoiceDate",
    "UnitPrice",
    "CustomerID",
    "Revenue"
]].isna().sum())

InvoiceNo      0
StockCode      0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Revenue        0
dtype: int64


In [43]:
print("Min InvoiceDate:", df_customer["InvoiceDate"].min())
print("Max InvoiceDate:", df_customer["InvoiceDate"].max())

Min InvoiceDate: 2010-12-01 08:26:00
Max InvoiceDate: 2011-12-09 12:50:00


In [44]:
print("=" * 50)
print("FINAL TRANSACTION DATASET")
print("=" * 50)

print(f"Rows       : {len(df_clean):,}")
print(f"Columns    : {len(df_clean.columns):,}")
print(f"Customers  : {df_clean['CustomerID'].nunique():,}")

print("\nMissing values:")
print(df_clean.isna().sum())

print("\nInvalid values:")
print(f"Quantity <= 0  : {(df_clean['Quantity'] <= 0).sum():,}")
print(f"UnitPrice <= 0 : {(df_clean['UnitPrice'] <= 0).sum():,}")
print(f"Revenue <= 0   : {(df_clean['Revenue'] <= 0).sum():,}")

FINAL TRANSACTION DATASET
Rows       : 524,878
Columns    : 10
Customers  : 4,338

Missing values:
InvoiceNo            0
StockCode            0
Description          0
Quantity             0
InvoiceDate          0
UnitPrice            0
CustomerID      132186
Country              0
is_cancelled         0
Revenue              0
dtype: int64

Invalid values:
Quantity <= 0  : 0
UnitPrice <= 0 : 0
Revenue <= 0   : 0


In [45]:
print("=" * 50)
print("FINAL CUSTOMER DATASET")
print("=" * 50)

print(f"Rows          : {len(df_customer):,}")
print(f"Unique Customers : {df_customer['CustomerID'].nunique():,}")

print("\nMissing values:")
print(df_customer.isna().sum())

print("\nInvalid values:")
print(f"Quantity <= 0  : {(df_customer['Quantity'] <= 0).sum():,}")
print(f"UnitPrice <= 0 : {(df_customer['UnitPrice'] <= 0).sum():,}")
print(f"Revenue <= 0   : {(df_customer['Revenue'] <= 0).sum():,}")

FINAL CUSTOMER DATASET
Rows          : 392,692
Unique Customers : 4,338

Missing values:
InvoiceNo       0
StockCode       0
Description     0
Quantity        0
InvoiceDate     0
UnitPrice       0
CustomerID      0
Country         0
is_cancelled    0
Revenue         0
dtype: int64

Invalid values:
Quantity <= 0  : 0
UnitPrice <= 0 : 0
Revenue <= 0   : 0


In [46]:
df_clean.to_csv(
    "../data/processed/online_retail_cleaned.csv",
    index=False
)

df_customer.to_csv(
    "../data/processed/online_retail_customer.csv",
    index=False
)

print("Processed datasets exported successfully.")

Processed datasets exported successfully.


## Data Cleaning Summary

The dataset underwent several cleaning and validation steps to prepare it for transaction analysis and customer segmentation.

### Cleaning Steps

1. **Removed duplicate rows**
   - Duplicate rows removed: 5,268
   - Remaining duplicate rows: 0

2. **Identified and removed cancelled transactions**
   - Cancelled transactions removed: 9,251
   - Cancellation rate before removal: 1.72%
   - Remaining cancelled transactions: 0

3. **Removed invalid UnitPrice values**
   - Rows with `UnitPrice <= 0` removed: 2,512
   - Remaining invalid UnitPrice values: 0

4. **Validated transaction metrics**
   - `Quantity <= 0`: 0
   - `UnitPrice <= 0`: 0
   - `Revenue <= 0`: 0

5. **Validated product descriptions**
   - Missing `Description`: 0
   - No imputation was required.

6. **Created customer-level transaction dataset**
   - Transactions with missing `CustomerID` were excluded from the customer-level dataset.
   - Customer-level dataset contains 4,338 unique customers.

### Final Dataset Summary

| Dataset | Rows | Unique Customers | Missing CustomerID | Purpose |
|---|---:|---:|---:|---|
| `df_clean` | 524,878 | 4,338 | 132,186 | Transaction & sales analysis |
| `df_customer` | 392,692 | 4,338 | 0 | RFM & customer segmentation |

### Final Data Quality

The cleaned transaction dataset contains no invalid quantity, unit price, or revenue values.

The customer-level dataset contains no missing values and is ready for customer-level analysis, including **RFM analysis and customer segmentation**.

### Business Interpretation

The cleaning process removed duplicate records, cancelled transactions, and invalid pricing records that could distort revenue and customer behavior analysis.

Transactions without a `CustomerID` were retained in the general transaction dataset because they may still be useful for overall sales analysis. However, they were excluded from the customer-level dataset because customer identification is required for RFM analysis and segmentation.

The resulting `df_customer` dataset provides a reliable foundation for analyzing customer purchasing behavior.